<a href="https://colab.research.google.com/github/shin-noda/leetcode-neetcode-250/blob/main/Problem1489.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))

        # Tracks number of connected components
        self.count = n


    def find(self, i):
        if self.parent[i] == i:
            return i

        # Path compression
        return self.find(self.parent[i])


    def union(self, i, j):
        root_i = self.find(i)
        root_j = self.find(j)

        if root_i != root_j:
            self.parent[root_i] = root_j
            self.count -= 1

            return True

        return False


class Solution:
    def findCriticalAndPseudoCriticalEdges(self, n, edges):
        # Use Kruskal's Algorithm

        # 1. Add original index to each edge so we can track them after sorting
        for i in range(len(edges)):
            edges[i].append(i)

        # 2. Sort edges by weight
        edges.sort(key=lambda x: x[2])

        # Helper function to find MST weight with modifications
        def get_mst_weight(skip_edge_idx=-1, force_edge_idx=-1):
            uf = UnionFind(n)
            weight = 0

            # If we are forcing an edge, add it first
            if force_edge_idx != -1:
                for src, dst, w, idx in edges:
                    if idx == force_edge_idx:
                        uf.union(src, dst)
                        weight += w
                        break

            # Run standard Kruskal's
            for src, dst, w, idx in edges:
                if idx == skip_edge_idx:
                    continue

                if uf.union(src, dst):
                    weight += w

            # If the graph isn't fully connected, return infinity
            if uf.count != 1:
                return float('inf')

            return weight


        # 3. Calculate the baseline MST weight
        base_weight = get_mst_weight()

        critical = []
        pseudo_critical = []

        # 4. Check every edge
        for src, dst, w, idx in edges:
            # Check if critical (skipping it makes MST worse)
            if get_mst_weight(skip_edge_idx=idx) > base_weight:
                critical.append(idx)

            # Check if pseudo-critical (Forcing it still yields an optimal MST)
            elif get_mst_weight(force_edge_idx=idx) == base_weight:
                pseudo_critical.append(idx)

        return [critical, pseudo_critical]